# 1. Data

The Yambda-50M listening log is turned into the interaction log that the later notebooks read. Events are split on
one time axis: the last $T_{\text{test}}$ days form the test period, the $T_{\text{val}}$ days before them the
validation period, everything earlier the training period. The catalogue consists of the most frequent items of
the training period, and item statistics are computed on the training period only.

Two retention signals are defined for an event at time $t$ from the times $t' \ge t$ of the same user's later events:

$$c = \mathbb{1}\big[\min_{t'}\,(t' - t) \le g\big], \qquad u = \mathbb{1}\big[\exists\, t' :\; g < t' - t \le w\big],$$

with session gap $g$ and return window $w$. An outcome counts as observed only if its waiting time ends inside the
observed data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gptrec.data import yambda
from gptrec.data.preprocess import PreprocessingConfig, build_interaction_log
from gptrec.paths import PROCESSED_DATA_DIR, RAW_DATA_DIR

In [ ]:
listens_path = yambda.download_listens(size="50m", directory=RAW_DATA_DIR)
raw_events = yambda.load_raw_events(listens_path)
print(f"{len(raw_events):,} raw events")
raw_events.head()

In [ ]:
config = PreprocessingConfig(max_items=30_000, min_item_events=20, min_user_events=50, validation_days=7, test_days=7)
log = build_interaction_log(raw_events, config).save(PROCESSED_DATA_DIR / "yambda-50m")

## Summary

In [ ]:
def num_events_in(period: str) -> int:
    starts, ends = log.period_rows(period)
    return int((ends - starts).sum())


def rate_where_observed(signal, is_observed) -> float:
    return float(np.asarray(signal)[np.asarray(is_observed)].mean())


pd.Series({
    "users": log.num_users,
    "items": log.num_items,
    "events": log.num_events,
    "train events": num_events_in("train"),
    "validation events": num_events_in("validation"),
    "test events": num_events_in("test"),
    "median events per user": float(np.median(np.diff(log.user_offsets))),
    "mean played ratio": float(np.mean(log.played_ratio)),
    "share of events played at least half": float(np.mean(np.asarray(log.played_ratio) >= 0.5)),
    "share of events that continue the session": rate_where_observed(log.session_continues, log.session_continues_known),
    "share of events after which the user returns": rate_where_observed(log.returns, log.returns_known),
    "share of organic events": float(np.mean(log.is_organic)),
})

## Played ratio

Distribution of the played ratio $\rho \in [0, 1]$, the response the recommender is later fine-tuned for, and the
mean response by the origin of the event: initiated by the user, or served by a recommender.

In [ ]:
sample = np.sort(np.random.default_rng(0).choice(log.num_events, size=min(log.num_events, 2_000_000), replace=False))
events = pd.DataFrame({
    "played_ratio": np.asarray(log.played_ratio)[sample],
    "session_continues": np.asarray(log.session_continues)[sample],
    "is_organic": np.asarray(log.is_organic)[sample],
})
events.played_ratio.plot.hist(bins=50, title="played ratio", figsize=(6, 3));

In [ ]:
events.groupby("is_organic")[["played_ratio", "session_continues"]].mean()

## Played ratio by item group

Mean played ratio of items by decile of duration and of popularity. A policy rewarded for $\rho$ can raise it by
moving to the groups with a high mean, without becoming more relevant.

In [ ]:
items = pd.DataFrame({
    "popularity": log.items.train_popularity[1:],
    "duration_seconds": log.items.duration_seconds[1:],
    "mean_played_ratio": log.items.mean_played_ratio[1:],
})
duration_decile = pd.qcut(items.duration_seconds, 10, duplicates="drop")
popularity_decile = pd.qcut(items.popularity.rank(method="first"), 10, labels=False)

figure, (left, right) = plt.subplots(1, 2, figsize=(13, 3.5))
items.groupby(duration_decile, observed=True).mean_played_ratio.mean().plot.bar(ax=left, title="mean played ratio by item duration (s)")
items.groupby(popularity_decile).mean_played_ratio.mean().plot.bar(ax=right, title="mean played ratio by popularity decile (9 = most popular)")
figure.tight_layout()